# Week 3 -- Strategy Reflection
### Imperial College -- Bayesian Optimisation Capstone

---

This notebook documents a critical reflection on the optimisation strategy used across Weeks 1-3,  
written after submitting the Week 3 queries and reviewing Week 2 results.

**Covers:**
- How the query strategy evolved across rounds
- Exploration vs exploitation balance
- How SVMs could complement the current approach
- Limitations of the GP model as data grows
- What this black-box setup teaches about real-world data science

---
## 1. How has my query strategy changed from earlier rounds?

Week 1 was straightforward -- I used the same UCB setup with beta 2.5 for all eight functions. I had no results yet, so treating every function the same made sense at the time.

After Week 2, that approach started to feel like a problem. Some functions came back better, some came back worse. Applying the same settings to all of them regardless of what the results were saying felt like ignoring the only information I actually had.

By Week 3 I was making different decisions per function based on what each one had shown:

| Function | Week 1 | Week 2 | Week 3 |
|----------|--------|--------|--------|
| F1 | UCB beta=2.5 | UCB beta=3.5 | UCB beta=3.5 |
| F2 | UCB beta=2.5 | UCB beta=3.5 | UCB beta=3.5 |
| F3 | UCB beta=2.5 | UCB beta=3.5 | UCB beta=3.5 |
| F4 | UCB beta=2.5 | UCB beta=3.5 | Switched to EI |
| F5 | UCB beta=2.5 | UCB beta=2.5 | UCB beta=2.0 |
| F6 | UCB beta=2.5 | UCB beta=3.5 | UCB beta=3.5 |
| F7 | UCB beta=2.5 | UCB beta=3.5 | UCB beta=2.0 |
| F8 | UCB beta=2.5 | UCB beta=4.0 | Switched to EI |

F4 got worse two weeks in a row which told me UCB was not the right approach there -- it kept sending the search into uncertain regions that turned out to be bad. Switching to EI made more sense because EI only recommends a point if it has a real chance of beating the current best, rather than just being uncertain. F5 and F7 improved noticeably so I pulled beta down to 2.0 to stay closer to what was already working rather than exploring further away.

Moving from applying the same settings to everything, to letting the results guide each decision -- that was the most significant shift across the three rounds.

---
## 2. How do I balance exploration against exploitation?

Week 1 was fully exploratory -- there were no results yet so exploring broadly was the only option. Week 2 showed that this was not always the right call. For most functions the returned outputs were worse than what the initial data had already found, which meant the GP was sending queries into unknown regions that turned out to be poor.

By Week 3 the balance shifted depending on what each function had shown:

- **F5 and F7** both improved clearly in Week 2. Continuing to explore away from those regions made no sense -- I dropped beta to 2.0 to stay close to the known good area and refine from there rather than wander off.
- **F1** came back near zero twice. There is nothing useful in the regions visited so far, so the only sensible move is to keep exploring and try somewhere completely different.
- **F4 and F8** kept returning results that were only marginally different from what was already known. UCB was spending queries on uncertain areas that were not actually better. Switching to EI forces the model to only recommend a point when it genuinely expects improvement over the current best.

The key thing I learned is that exploration is only worth doing if you have not yet found a good region. Once a function has shown a clear peak or a consistently improving area, staying near that region is more useful than chasing uncertainty elsewhere -- especially with only one query per week.

| Function | Status after Week 2 | Week 3 approach |
|----------|---------------------|-----------------|
| F1 | Near zero both times | Keep exploring |
| F2 | Small improvement | Keep exploring |
| F3 | Slowly improving | Keep exploring |
| F4 | Worse two weeks running | Switch to EI |
| F5 | Clear improvement | Exploit -- stay near peak |
| F6 | Slightly worse | Keep exploring |
| F7 | Clear improvement | Exploit -- stay near peak |
| F8 | Marginal gains only | Switch to EI |

---
## 3. How would SVMs change my approach?

After three weeks I have a set of observed inputs and outputs for each function. One way to use that data differently would be to label each observation as either good or poor based on a threshold -- for example anything in the top 30 percent of observed outputs gets labelled as good, everything else as poor. A soft-margin SVM could then learn a boundary between those two groups in the input space.

That boundary could act as a filter. Instead of letting the GP search the entire input space, only the region the SVM classifies as good would be passed to the acquisition function. This would cut out areas the data has already shown to be poor and focus the search on where results have actually been promising.

A simple linear SVM would probably not work well here because the good regions in most of these functions are not shaped like a flat plane -- they are more like a blob or a curved patch in the space. An RBF kernel SVM would handle that better since it can draw curved, more flexible boundaries.

The limitation is that with only 12 to 42 observations per function, there might not be enough labelled examples for the SVM to learn a reliable boundary. It would become more useful as more data accumulates over future weeks.

In short, the SVM would not replace the GP -- it would sit in front of it as a way of narrowing down where the GP needs to look, which would be especially helpful for F7 and F8 where the input space is very large and most of it is probably uninteresting.

---
## 4. What limitations has my model shown as data grows?

**The kernel settings are fixed and might not fit every function.**
The GP uses an RBF kernel with a fixed length scale of 0.1. This setting controls how quickly the model thinks the function changes as you move through the input space. If the real function changes more slowly or quickly than that, the GP builds a misleading picture and its recommendations become unreliable. With more data coming in, this mismatch becomes more visible.

**Early bad results can mislead the GP for several rounds.**
F4 is the clearest example. The early queries landed in a poor region and the GP fitted its model around those bad observations. Every suggestion it made after that was influenced by that early cluster. It took multiple rounds just to correct that initial bias rather than making progress toward the actual maximum.

**Not all input dimensions matter equally, but the GP treats them the same.**
In F8 with 8 inputs, it is very likely that some dimensions have much more influence on the output than others. The GP has no way to figure that out with a fixed kernel -- it gives equal weight to all dimensions regardless. This wastes part of the search effort on dimensions that probably do not matter much locally.

**The data is very thin relative to the number of dimensions in higher-dimensional functions.**

| Function | Dimensions | Observations after Week 3 | Observations per dimension |
|----------|-----------|--------------------------|---------------------------|
| F1 | 2 | 12 | 6 |
| F7 | 6 | 32 | ~5 |
| F8 | 8 | 42 | ~5 |

F8 has 8 input dimensions and only 42 observations total. That is not much to build a reliable surface map from, and the GP is essentially guessing across most of the space.

---
## 5. How does this black-box setup prepare me to think like a data scientist?

The biggest thing this challenge made clear is that in real projects you rarely get to see the full picture before making a decision. You get some data, you make your best call with it, you wait for the result, and then you update. That is exactly what this setup forces you to do -- one query per week, no peeking at the function, no do-overs.

What I found useful is that a bad result is not a wasted week. When F4 came back at -24 instead of the expected improvement, that told me something concrete -- that region of the input space is genuinely poor and the GP had been wrong about it. That is information. It narrowed down where the maximum is not, which is still progress.

The shift from UCB to EI for F4 and F8 felt like a small technical change but it reflected something more important -- the difference between acting on uncertainty alone and acting on evidence. UCB sends you to places the model has not seen yet. EI sends you to places the model thinks are genuinely better than what you already have. Knowing when to use which one is not something you can decide upfront. It depends on what the results are actually telling you.

That judgment call -- when to trust the model, when to question it, and when to change the approach entirely -- is something that comes up constantly in real data science work. This project gave me a way to practise it with actual consequences rather than just reading about it.

---

*Reflection written after Week 3 submission | Imperial College Bayesian Optimisation Capstone*